# Chapter 5 &mdash; LSB-First "Divisible by 3": a Pair-Valued State

**Concept 9 of the Chapter 5 decomposition:** *LSB-First "Divisible by 3": a Pair-Valued State*

Each new bit lands leftmost, so the state must track the running weight $2^k$ as well as the residue.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-LSB-First-Pair-State/Concept-LSB-First-Pair-State.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Reading **least-significant bit first** changes everything: bit $b$ arriving at
position $k$ contributes $b\cdot 2^k$, so the value update is
$N \mapsto N + b\cdot 2^k$ and you need to know $2^k$.

You still cannot store $k$ &mdash; but you can store $2^k \bmod 3$, which cycles
$1, 2, 1, 2, \dots$

So the state is a **pair** $(N\bmod 3,\ 2^k\bmod 3)$: six combinations, of which the
reachable ones form the machine. Same language as the MSB machine, different design.

## 2. Definitions

### The pair-valued recurrence

In [ ]:
def lsb_step(state, b):
    r, w = state                      # (value mod 3, current weight mod 3)
    return ((r + int(b)*w) % 3, (2*w) % 3)

def lsb_resid(s):
    st = (0, 1)
    for b in s: st = lsb_step(st, b)
    return st[0]

### The DFA over pairs, names spelling out the pair

In [ ]:
# Naming: exactly ONE state may begin with 'I'.  Both r=0 states are final,
# so the start state is IF_r0w1 and the other final state is plain F_r0w2.
Div3L = md2mc('''DFA
IF_r0w1 : 0 -> F_r0w2       !! r stays 0, weight 1->2
IF_r0w1 : 1 -> S_r1w2       !! r += 1*1
F_r0w2  : 0 -> IF_r0w1
F_r0w2  : 1 -> S_r2w1       !! r += 1*2
S_r1w2  : 0 -> S_r1w1
S_r1w2  : 1 -> IF_r0w1      !! 1 + 2 = 3 = 0 mod 3
S_r1w1  : 0 -> S_r1w2
S_r1w1  : 1 -> S_r2w2
S_r2w1  : 0 -> S_r2w2
S_r2w1  : 1 -> F_r0w2       !! 2 + 1 = 3 = 0 mod 3
S_r2w2  : 0 -> S_r2w1
S_r2w2  : 1 -> S_r1w1       !! 2 + 2 = 4 = 1 mod 3
''')

### Reference: interpret the string LSB-first

In [ ]:
def val_lsb(s): return int(s[::-1], 2) if s else 0

<!-- nav-strip -->

---

&larr;&nbsp;[Ch5&nbsp;8.&nbsp;The Mod Algebra That Makes Residue States Computable](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Mod-Algebra/Concept-Mod-Algebra.ipynb) &nbsp;&middot;&nbsp; [**Chapter 5** index](https://github.com/ganeshutah/Jove/blob/master/Chapter5-DFADsg/README.md) &nbsp;&middot;&nbsp; [Ch5&nbsp;10.&nbsp;Exponential Blow-Up: "Third-Last Bit is a 1"](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5-DFADsg/Concept-Exponential-Blow-Up/Concept-Exponential-Blow-Up.ipynb)&nbsp;&rarr;

---

## 3. Tests

The pair recurrence computes the right residue.

In [ ]:
from itertools import product
bad = [''.join(p) for k in range(1, 12) for p in product('01', repeat=k)
       if lsb_resid(''.join(p)) != val_lsb(''.join(p)) % 3]
print("mismatches :", bad)
assert not bad
print("weight cycle 2^k mod 3 :", [(2**k) % 3 for k in range(8)], " <- period 2")

And the DFA recognises LSB-first divisibility by 3.

In [ ]:
bad = [''.join(p) for k in range(1, 12) for p in product('01', repeat=k)
       if accepts_dfa(Div3L, ''.join(p)) != (val_lsb(''.join(p)) % 3 == 0)]
print("DFA mismatches on 1..11-bit LSB-first numerals :", len(bad))
assert not bad
for s in ['0', '11', '011', '0011']:
    print("  %-6r reads as %-4d divisible by 3? %s"
          % (s, val_lsb(s), accepts_dfa(Div3L, s)))

Six states here versus three for MSB-first &mdash; but minimization tells the real story.

In [ ]:
mL = min_dfa(Div3L)
print("LSB machine: %d states, minimized to %d" % (len(Div3L["Q"]), len(mL["Q"])))
print("\nThe weight component is genuinely needed: reading direction changes the language")
print("of *strings* even though it is the same set of *numbers*.")

## 4. Animation

Follow the pair: the second component alternates every step, the first accumulates.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(Div3L, FuseEdges=True)

## 5. Exercises


1. Which pairs are unreachable, and why?
2. Build the LSB-first divisible-by-5 machine. What is the weight cycle length?
3. Are the MSB and LSB machines language-equivalent? Check with `langeq_dfa` and explain.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter5-DFADsg/Concept-LSB-First-Pair-State')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')